In [62]:
import pandas as pd

roles_df = pd.read_excel("../data/company_roles/company_roles.xlsx")
roles_df.head()

,company,role,description,required_skill,department
0,Mekari,AI and Data Specialist,The AI/Data Builder is an AI-native Builder wh...,2 - 3+ years in full-stack engineering or appl...,Information Technology
1,Mekari,Junior Payroll Specialist,Collect and validate employee time and attenda...,"Bachelor's degree in management, HR, or a rela...",Human Resource
2,Mekari,Technical Support,"Coordinate with Account Manager, Customer Supp...",Candidate must possess at least a Bachelor's D...,Information Technology
3,Mekari,SEO Associate,Create and implement comprehensive SEO Content...,At least 2 years experience in optimize SEO ca...,Marketing
4,Mekari,Solution Engineering,The Solution Engineer is responsible for worki...,Educational background in Information Technolo...,Information Technology


In [63]:
roles_df.shape

(9, 5)

In [64]:
roles_df.isnull().sum()

company           0
role              0
description       0
required_skill    0
department        0
dtype: int64

In [65]:
roles_df.duplicated().sum()

0

In [66]:
# make new column 

roles_df["embedding_text"] = (
    roles_df["role"] + ". "
    + roles_df["description"] + ". "
    + roles_df["required_skill"] + ". "
    + roles_df["department"]
)

In [67]:
roles_df[["role", "embedding_text"]].head()

,role,embedding_text
0,AI and Data Specialist,AI and Data Specialist. The AI/Data Builder is...
1,Junior Payroll Specialist,Junior Payroll Specialist. Collect and validat...
2,Technical Support,Technical Support. Coordinate with Account Man...
3,SEO Associate,SEO Associate. Create and implement comprehens...
4,Solution Engineering,Solution Engineering. The Solution Engineer is...


In [68]:
from sentence_transformers import SentenceTransformer

In [69]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [70]:
roles_df["embedding"] = roles_df["embedding_text"].apply(
    lambda x : embedding_model.encode(
        x,
        show_progress_bar=False
    )
)

In [71]:
roles_df["embedding"].head(1)

0    [-0.039885316, -0.057568796, 0.03350795, 0.017...
Name: embedding, dtype: object

In [72]:
roles_df["embedding"].iloc[0].shape

(384,)

In [73]:
# cek embedding seluruh role

roles_df["embedding"].apply(lambda x : x.shape).value_counts()

embedding
(384,)    9
Name: count, dtype: int64

In [74]:
roles_df[[
    "company",
    "role",
    "department",
    "embedding_text",
    "embedding"
]].head(2)

,company,role,department,embedding_text,embedding
0,Mekari,AI and Data Specialist,Information Technology,AI and Data Specialist. The AI/Data Builder is...,"[-0.039885316, -0.057568796, 0.03350795, 0.017..."
1,Mekari,Junior Payroll Specialist,Human Resource,Junior Payroll Specialist. Collect and validat...,"[-0.07262973, 0.111312315, -0.010233567, -0.01..."


In [75]:
role_data = []

for _, row in roles_df.iterrows():
    role_data.append({
        "company": row["company"],
        "role": row["role"],
        "description": row["description"],
        "required_skill": row["required_skill"],
        "department": row["department"],
        "embedding": row["embedding"].tolist()
    })

In [76]:
# konek ke supabase
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv("../.env")

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

supabase = create_client(
    supabase_url,
    supabase_key
)

In [77]:
# insert data ke supabase

response = supabase.table("company_roles").insert(role_data).execute()

In [78]:
# panggil data dari supabase untuk verifikasi
# response.data -> ambil bagian data dari response tersebut.
# hasil dari supabase adalah list of dictionaries.
# untuk dapat melihatnya dalam bentuk table, makanya di konversi ke DF

response = supabase.table("company_roles").select(
    "id, company, role, department"
).execute()

pd.DataFrame(response.data)

,id,company,role,department
0,1,Mekari,AI and Data Specialist,Information Technology
1,2,Mekari,Junior Payroll Specialist,Finance
2,3,Mekari,Technical Support,Finance
3,4,Mekari,SEO Associate,Marketing
4,5,Mekari,Solution Engineering,Information Technology
5,6,Mekari,Risk Management Specialist,Legal
6,7,Mekari,Project Manager,Marketing
7,8,Mekari,IT Auditor Lead,Information Technology
8,9,Mekari,Senior QA Engineer,Information Technology
9,15,Mekari,Risk Management Specialist,Legal


# Semantic Matching

In [79]:
from pypdf import PdfReader

pdf_path = "../data/Curriculum Vitae Syarifah Kemala Putri.pdf"

reader = PdfReader(pdf_path)

sample_cv = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        sample_cv += page_text +"\n"

In [80]:
print(sample_cv[:1000])

SYARIFAH KEMALA PUTRI 
  +6281351952909 |  Medan, Indonesia |    syarifahkemalaputri@gmail.com 
 
PROFESSIONAL SUMMARY
 
Computer Science graduate with experience in data processing, web scraping validation, and NLP research. 
Skilled in Python, data analysis, and working with structured datasets. Proven ability to maintain high data 
quality through QC processes and consistently achieve strong KPI performance. 
 
WORK EXPERIENCE
 
Website Administrator | Faculty of Engineering USU   
September 2019 – September 2020 
o Responsible for publishing the latest news, circulars, and activities related to the faculty. 
o Responsible for organizing activity publication documents and softcopy of circulars on campus. 
o Delivering the latest information and upcoming activities to students. 
 
Intern | KPKNL, Kementrian Keuangan Republik Indonesia  
July 2019 – August 2019 
o Intern as Web Developer in General Subsection division to build Archiving information system using 
HTML, CSS, Javascript 

In [81]:
cv_embedding = embedding_model.encode(
    sample_cv,
    show_progress_bar=False
)

cv_embedding.shape

(384,)

In [82]:
response = supabase.rpc(
    "match_company_roles",
    {
        "query_embedding": cv_embedding.tolist(),
        "match_count": 5
    }
).execute()

In [83]:
results_df = pd.DataFrame(response.data)

results_df[[
    "role",
    "department",
    "similarity"
]]

,role,department,similarity
0,Senior QA Engineer,Information Technology,0.373487
1,Senior QA Engineer,Information Technology,0.373487
2,Technical Support,Information Technology,0.341405
3,AI and Data Specialist,Information Technology,0.341156
4,AI and Data Specialist,Information Technology,0.341156


# Skill Dictionary

In [84]:
for _, row in roles_df.iterrows():
    print("=" * 80)
    print(row["role"])
    print("=" * 80)
    print(row["required_skill"])
    print()

AI and Data Specialist
2 - 3+ years in full-stack engineering or applied software development using AI/data use cases. Experience with AI, ML, LLM, RAG, data, analytics, and intelligent product systems. Proficiency in prompt, context, retrieval, model, orchestration, and data-system design.
Experience with data pipelines, data quality, freshness, lineage, completeness, and correctness. Demonstrated ability with evaluation design, golden datasets, regression evals, scenario tests, and human review. Working knowledge of grounding, hallucination control, refusal behavior, guardrails, safety, and trust.
Experience monitoring for drift, quality degradation, latency, cost, user feedback, and production behavior. Track record shipping AI/data-powered products, features, modules, workflows, services, or decision-support capabilities. Preferred Traits, Experience collaborating with Platform Builders on agentic build workflows, especially agent tuning, evaluation, and improvement. Comfort defini

In [85]:
# generate skill dictionary using LLM

import os 
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENROUTER_API_KEY")

print("API key tersedia", api_key is not None)


API key tersedia True


In [86]:
import requests
import json

def extract_skill(required_skill):
    prompt= f""" You are an expert HR and technical recruiter.

Your task is to extract the skills and competencies required for a job
from the job requirement text below.

Extract only items that can reasonably be considered a candidate skill,
technical competency, professional competency, technology, tool,
methodology, or domain-specific expertise.

Rules:

1. Prefer concise skill names rather than long phrases.
2. Normalize similar concepts into one standard skill.
3. Include technical skills, tools, technologies, methodologies,
   professional competencies, and domain knowledge.
4. Do NOT include:
   - years of experience
   - education or degrees
   - generic words such as "experience", "ability", "strong"
   - personality traits
   - job responsibilities
   - ownership statements
   - business outcomes
   - complete sentences
   - vague phrases
5. Do not extract a responsibility such as:
   "owning AI quality"
   "shipping products"
   "collaborating with teams"
6. Instead, extract the underlying skill when it is explicitly stated.
   For example:
   "design and evaluate AI systems"
   → "AI system design", "AI evaluation"
7. Combine duplicate or highly similar skills.
8. Keep the skill names concise, normally 1–4 words.
9. Return approximately 15–30 of the most relevant skills.
10. Return ONLY a valid JSON array of strings.
11. Do not include markdown, explanations, or additional text.

Job requirement:
{required_skill}"""
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Beare {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
            "messages": [
                {
                "role": "user",
                "content": prompt
                }
            ],
            "temperature": 0
        }
    )
    response.raise_for_status()

    result = response.json()
    content = result["choices"][0]["message"]["content"]

    return json.loads(content)


In [87]:
required_skill = roles_df.loc[0, "required_skill"]

In [89]:
skill = extract_skill(required_skill)
skill

['Full-stack engineering',
 'Machine Learning',
 'Large Language Models',
 'Retrieval-Augmented Generation',
 'Prompt engineering',
 'Retrieval systems',
 'Model orchestration',
 'Data system design',
 'Data pipelines',
 'Data quality',
 'Data lineage',
 'Evaluation design',
 'Golden datasets',
 'Regression evaluation',
 'Scenario testing',
 'Human review',
 'Grounding',
 'Hallucination control',
 'Guardrails',
 'AI safety',
 'Model monitoring',
 'Drift detection',
 'Latency monitoring',
 'Agentic workflows',
 'Agent tuning',
 'Agent evaluation',
 'AI quality management',
 'Production AI systems']

In [90]:
import requests
import json

def extract_skills(required_skill):

    prompt = f"""
You are an expert HR and technical recruiter.

Your task is to extract the skills and competencies required for a job
from the job requirement text below.

Extract only items that can reasonably be considered a candidate skill,
technical competency, professional competency, technology, tool,
methodology, or domain-specific expertise.

Rules:

1. Prefer concise skill names rather than long phrases.
2. Normalize similar concepts into one standard skill.
3. Include technical skills, tools, technologies, methodologies,
   professional competencies, and domain knowledge.
4. Do NOT include:
   - years of experience
   - education or degrees
   - generic words such as "experience", "ability", "strong"
   - personality traits
   - job responsibilities
   - ownership statements
   - business outcomes
   - complete sentences
   - vague phrases
5. Do not extract a responsibility such as:
   "owning AI quality"
   "shipping products"
   "collaborating with teams"
6. Instead, extract the underlying skill when it is explicitly stated.
7. Combine duplicate or highly similar skills.
8. Keep the skill names concise, normally 1–4 words.
9. Return approximately 15–30 of the most relevant skills.
10. Return ONLY a valid JSON array of strings.
11. Do not include markdown, explanations, or additional text.

Job requirement:
{required_skill}
"""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0
        }
    )

    # Kalau request gagal
    if response.status_code != 200:
        print("Status code:", response.status_code)
        print("Response:", response.text)
        return None

    result = response.json()

    # Pastikan response memiliki choices
    if "choices" not in result:
        print("Response tidak memiliki 'choices':")
        print(result)
        return None

    content = result["choices"][0]["message"]["content"]

    # Bersihkan markdown JSON jika model menambahkannya
    content = content.strip()

    if content.startswith("```"):
        content = content.replace("```json", "")
        content = content.replace("```", "")
        content = content.strip()

    return json.loads(content)

In [91]:
required_skill = roles_df.loc[4, "required_skill"]

skills = extract_skills(required_skill)

skills

Response tidak memiliki 'choices':
{'id': 'gen-1791277016-fKv6WGBmRTGcBmhAE4KZ', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}


In [92]:
import time

roles_df["extracted_skills"] = None

for i, row in roles_df.iterrows():

    print(f"Processing {i+1}/{len(roles_df)}: {row['role']}")

    skills = extract_skills(row["required_skill"])

    if skills is not None:
        roles_df.at[i, "extracted_skills"] = skills

    time.sleep(3)

Processing 1/9: AI and Data Specialist
Processing 2/9: Junior Payroll Specialist
Processing 3/9: Technical Support
Processing 4/9: SEO Associate
Processing 5/9: Solution Engineering
Response tidak memiliki 'choices':
{'id': 'gen-1791277267-e1RucNCXmjETLUO3UBVM', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}
Processing 6/9: Risk Management Specialist
Processing 7/9: Project Manager
Processing 8/9: IT Auditor Lead
Processing 9/9: Senior QA Engineer
Response tidak memiliki 'choices':
{'id': 'gen-1791277546-1LxbOycgKu9t1YFEjc20', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}


In [93]:
roles_df[["role","extracted_skills"]]

,role,extracted_skills
0,AI and Data Specialist,"[Full-stack engineering, Artificial Intelligen..."
1,Junior Payroll Specialist,"[Payroll Processing, Payroll Compliance, Emplo..."
2,Technical Support,"[Information Systems, Logistics Management, Ac..."
3,SEO Associate,"[SEO Campaign Optimization, SEO Principles, AI..."
4,Solution Engineering,None
5,Risk Management Specialist,"[Risk Management, SaaS Domain Knowledge, Stake..."
6,Project Manager,"[Project Management, ERP Implementation, ERP C..."
7,IT Auditor Lead,"[IT Audit, IT Risk Management, Cybersecurity, ..."
8,Senior QA Engineer,None


In [98]:
# retry 2 role yang gagal

import time

failed_roles = roles_df[
    roles_df["extracted_skills"].isna()
]

for i, row in failed_roles.iterrows():

    print(f"Retry: {row['role']}")

    skills = extract_skills(row["required_skill"])

    if skills is not None:
        roles_df.at[i, "extracted_skills"] = skills
        print("Berhasil")
    else:
        print("Masih gagal")

    time.sleep(10)

Retry: Solution Engineering
Berhasil
Retry: Senior QA Engineer
Response tidak memiliki 'choices':
{'id': 'gen-1791277681-5hYXDif9YEknQerbk4Lr', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}
Masih gagal


In [99]:
roles_df[["role", "extracted_skills"]]

,role,extracted_skills
0,AI and Data Specialist,"[Full-stack engineering, Artificial Intelligen..."
1,Junior Payroll Specialist,"[Payroll Processing, Payroll Compliance, Emplo..."
2,Technical Support,"[Information Systems, Logistics Management, Ac..."
3,SEO Associate,"[SEO Campaign Optimization, SEO Principles, AI..."
4,Solution Engineering,"[JavaScript, CSS, REST API, SQL, Database Mana..."
5,Risk Management Specialist,"[Risk Management, SaaS Domain Knowledge, Stake..."
6,Project Manager,"[Project Management, ERP Implementation, ERP C..."
7,IT Auditor Lead,"[IT Audit, IT Risk Management, Cybersecurity, ..."
8,Senior QA Engineer,None


In [100]:
# retry 1 role yang gagal

import time

failed_roles = roles_df[
    roles_df["extracted_skills"].isna()
]

for i, row in failed_roles.iterrows():

    print(f"Retry: {row['role']}")

    skills = extract_skills(row["required_skill"])

    if skills is not None:
        roles_df.at[i, "extracted_skills"] = skills
        print("Berhasil")
    else:
        print("Masih gagal")

    time.sleep(5)

Retry: Senior QA Engineer
Berhasil


In [101]:
roles_df[["role", "extracted_skills"]]

,role,extracted_skills
0,AI and Data Specialist,"[Full-stack engineering, Artificial Intelligen..."
1,Junior Payroll Specialist,"[Payroll Processing, Payroll Compliance, Emplo..."
2,Technical Support,"[Information Systems, Logistics Management, Ac..."
3,SEO Associate,"[SEO Campaign Optimization, SEO Principles, AI..."
4,Solution Engineering,"[JavaScript, CSS, REST API, SQL, Database Mana..."
5,Risk Management Specialist,"[Risk Management, SaaS Domain Knowledge, Stake..."
6,Project Manager,"[Project Management, ERP Implementation, ERP C..."
7,IT Auditor Lead,"[IT Audit, IT Risk Management, Cybersecurity, ..."
8,Senior QA Engineer,"[Selenium, Playwright, Cypress, Appium, Python..."


In [ ]:
for _, row in roles_df.iterrows():

    print("=" * 60)
    print(row["role"])
    print("=" * 60)

    for skill in row["extracted_skills"]:
        print("-", skill)

    print()

AI and Data Specialist
- Full-stack engineering
- Artificial Intelligence (AI)
- Machine Learning (ML)
- Large Language Models (LLM)
- Retrieval-Augmented Generation (RAG)
- Prompt engineering
- Retrieval systems
- Orchestration
- Data system design
- Data pipelines
- Data quality
- Data lineage
- Evaluation design
- Golden datasets
- Regression evaluation
- Scenario testing
- Grounding
- Hallucination control
- Guardrails
- AI safety
- Model monitoring
- Drift detection
- Production monitoring
- Agentic workflows
- Agent optimization
- Non-deterministic systems
- AI quality management
- Production AI systems

Junior Payroll Specialist
- Payroll Processing
- Payroll Compliance
- Employment Tax Law
- HRIS Administration
- Payroll Software
- Talenta
- Microsoft Excel
- Spreadsheet Management
- Numerical Analysis
- Analytical Thinking
- Attention to Detail
- Confidentiality Management
- Data Privacy
- Communication Skills
- Interpersonal Skills
- Organizational Skills
- Time Management
- 

In [104]:
roles_df.to_excel("../data/company_roles/comopany_roles_with_skills.xlsx", index=False)

In [105]:
skill_rows = []

for _, row in roles_df.iterrows():
    for skill in row["extracted_skills"]:
        skill_rows.append({
            "role": row["role"],
            "department": row["department"],
            "skill": skill
        })
skills_df = pd.DataFrame(skill_rows)
skills_df.head(20)

,role,department,skill
0,AI and Data Specialist,Information Technology,Full-stack engineering
1,AI and Data Specialist,Information Technology,Artificial Intelligence
2,AI and Data Specialist,Information Technology,Machine Learning
3,AI and Data Specialist,Information Technology,Large Language Models
4,AI and Data Specialist,Information Technology,Retrieval-Augmented Generation
5,AI and Data Specialist,Information Technology,Prompt engineering
6,AI and Data Specialist,Information Technology,Context engineering
7,AI and Data Specialist,Information Technology,Retrieval systems
8,AI and Data Specialist,Information Technology,Model orchestration
9,AI and Data Specialist,Information Technology,Data system design


In [106]:
skills_df.groupby("role").size()

role
AI and Data Specialist        29
IT Auditor Lead               28
Junior Payroll Specialist     20
Project Manager               24
Risk Management Specialist    18
SEO Associate                 16
Senior QA Engineer            24
Solution Engineering          10
Technical Support              8
dtype: int64

In [107]:
skills_df.shape

(177, 3)